# NMPC

Step 9.1. The Phase 5 model in CasADi for acados (`ackermann_nmpc/model.py`): path coordinates
at the CG, the two /drive commands as states and their rates as inputs, friction `mu_scale` and
path curvature as parameters. Checked here against `scripts/vehicle_model.py`.

In [1]:
import os
import sys
import tempfile
sys.path.append('../scripts')
sys.path.append('../ros2_ws/src/ackermann_nmpc')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from acados_template import AcadosSim, AcadosSimSolver
import vehicle_model as vm
from ackermann_nmpc.model import TAU_U, forces, path_model

plt.rcParams['figure.dpi'] = 72
p = yaml.safe_load(open('../ros2_ws/src/ackermann_description/config/vehicle_params.yaml'))['/**']['ros__parameters']

/home/pietro/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


## Forces

Both at 10000 random states, over and past the range of the Phase 5 runs.

In [2]:
rng = np.random.default_rng(0)
n = 10000
X = np.c_[rng.uniform(0.1, 1.5, n), rng.uniform(-0.6, 0.6, n), rng.uniform(-5, 5, n),
          rng.uniform(-0.5, 0.5, n), rng.uniform(0, 1.3, n)]
ref = np.array([vm.forces(x[:4], x[4]) for x in X])
ours = np.array([np.array([float(v) for v in forces(p, *x, 1.0)]) for x in X])
err = np.abs(ours - ref).max(axis=0)
print(f'max |difference|: fx {err[0]:.1e} N, fy {err[1]:.1e} N, mz {err[2]:.1e} N m '
      f'(fy up to {np.abs(ref[:, 1]).max():.1f} N)')

max |difference|: fx 1.4e-07 N, fy 1.4e-07 N, mz 1.3e-08 N m (fy up to 6.4 N)


## Integration

The /drive commands of val_drift (0.9 m/s at full lock, the 1.3 m/s donut, 0.6 m/s) from 0.3 m/s:
the acados integrator (explicit RK4, 1 ms steps) against the numpy RK4 of the notebooks, same
1 ms. Once along a straight path, where s, n and e_psi are x, y and yaw, once along a circle of
5 m radius turning left, mapped back to x, y.

In [3]:
run = pd.read_parquet('../data/sysid/val_drift.parquet')
run = run[run['cmd_speed'].ffill() > 0].reset_index(drop=True)
t = run['t'].to_numpy()
d_cmd = np.interp(run['cmd_steer'].ffill(), vm.CMD, vm.DEL)
u_cmd = run['cmd_speed'].ffill().to_numpy()
dt = 0.02
steps = int((t[-1] - t[0]) / dt)
cmd = [np.searchsorted(t, t[0] + k * dt, side='right') - 1 for k in range(steps)]


def deriv(x, dc, uc):
    X, Y, psi, vx, vy, r, d, u = x
    fx, fy, mz = vm.forces((vx, vy, r, d), u)
    return np.array([vx * np.cos(psi) - vy * np.sin(psi), vx * np.sin(psi) + vy * np.cos(psi), r,
                     fx / vm.M + vy * r, fy / vm.M - vx * r, mz / vm.IZ, (dc - d) / vm.TAU, (uc - u) / TAU_U])


x = np.array([0, 0, 0, 0.3, 0, 0, d_cmd[0], 0.3])
ref = []
for i in cmd:
    h = dt / 20
    for _ in range(20):
        k1 = deriv(x, d_cmd[i], u_cmd[i])
        k2 = deriv(x + h / 2 * k1, d_cmd[i], u_cmd[i])
        k3 = deriv(x + h / 2 * k2, d_cmd[i], u_cmd[i])
        k4 = deriv(x + h * k3, d_cmd[i], u_cmd[i])
        x = x + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
    ref.append(x)
ref = np.array(ref)


def acados_run(kappa):
    sim = AcadosSim()
    sim.model = path_model(p)
    sim.parameter_values = np.array([1.0, kappa])
    sim.solver_options.T = dt
    sim.solver_options.integrator_type = 'ERK'
    sim.solver_options.num_stages = 4
    sim.solver_options.num_steps = 20
    sim.code_export_directory = tempfile.mkdtemp()
    solver = AcadosSimSolver(sim, json_file=os.path.join(sim.code_export_directory, 'sim.json'), verbose=False)
    x = np.array([0, 0, 0, 0.3, 0, 0, d_cmd[0], 0.3, d_cmd[0], u_cmd[0]])
    out = []
    for i in cmd:
        x[8:] = d_cmd[i], u_cmd[i]
        x = solver.simulate(x=x, u=np.zeros(2))
        out.append(x)
    return np.array(out)


rows = []
for kappa in [0.0, 0.2]:
    s, n, e_psi = acados_run(kappa)[:, :3].T
    if kappa == 0:
        X, Y, psi = s, n, e_psi
    else:
        R, a = 1 / kappa, s * kappa     # circle centred at (0, R), from the origin heading +x
        X, Y, psi = R * np.sin(a) - n * np.sin(a), R - R * np.cos(a) + n * np.cos(a), a + e_psi
    rows.append({'path': 'straight' if kappa == 0 else 'circle R 5 m',
                 'position [m]': np.hypot(X - ref[:, 0], Y - ref[:, 1]).max(),
                 'heading [rad]': np.abs(psi - ref[:, 2]).max()})
print(f'{steps * dt:.1f} s, yaw rate up to {ref[:, 5].max():.2f} rad/s, max |difference|:')
pd.DataFrame(rows)

/tmp/ipykernel_475543/4168213519.py:40: DeprecationWarning: Call to deprecated function (or staticmethod) code_export_directory. (Use AcadosSim.code_gen_options.code_export_directory instead.) -- Deprecated since version 0.5.4.
  sim.code_export_directory = tempfile.mkdtemp()
/tmp/ipykernel_475543/4168213519.py:41: DeprecationWarning: Call to deprecated function (or staticmethod) code_export_directory. (Use AcadosSim.code_gen_options.code_export_directory instead.) -- Deprecated since version 0.5.4.
  solver = AcadosSimSolver(sim, json_file=os.path.join(sim.code_export_directory, 'sim.json'), verbose=False)
/tmp/ipykernel_475543/4168213519.py:41: DeprecationWarning:  The `json_file` argument is deprecated in v0.5.6 and will be removed in a future release. Set AcadosSim.code_gen_options.json_file instead.
  solver = AcadosSimSolver(sim, json_file=os.path.join(sim.code_export_directory, 'sim.json'), verbose=False)
/home/pietro/acados/interfaces/acados_template/acados_template/utils.py:14

/tmp/ipykernel_475543/4168213519.py:40: DeprecationWarning: Call to deprecated function (or staticmethod) code_export_directory. (Use AcadosSim.code_gen_options.code_export_directory instead.) -- Deprecated since version 0.5.4.
  sim.code_export_directory = tempfile.mkdtemp()
/tmp/ipykernel_475543/4168213519.py:41: DeprecationWarning: Call to deprecated function (or staticmethod) code_export_directory. (Use AcadosSim.code_gen_options.code_export_directory instead.) -- Deprecated since version 0.5.4.
  solver = AcadosSimSolver(sim, json_file=os.path.join(sim.code_export_directory, 'sim.json'), verbose=False)
/tmp/ipykernel_475543/4168213519.py:41: DeprecationWarning:  The `json_file` argument is deprecated in v0.5.6 and will be removed in a future release. Set AcadosSim.code_gen_options.json_file instead.
  solver = AcadosSimSolver(sim, json_file=os.path.join(sim.code_export_directory, 'sim.json'), verbose=False)
/home/pietro/acados/interfaces/acados_template/acados_template/utils.py:14

16.0 s, yaw rate up to 5.49 rad/s, max |difference|:


,path,position [m],heading [rad]
0,straight,9.010355e-08,1.495347e-07
1,circle R 5 m,9.010407e-08,1.495489e-07


## Result

The CasADi model is the Phase 5 model: forces within 1.4e-7 N (the smoothing of the slip at zero),
and 16 s of val_drift through the acados integrator within 1e-7 m of the numpy one, along a
straight path and along a circle.